# Notebook 01 — SOP Stratified Sampling: 120K → 20K

## Mục tiêu

Notebook này **chỉ thực hiện sampling**, không chạy model/retrieval.

Pipeline:

`SOP 120K → đọc metadata → kiểm tra phân bố → lọc class hợp lệ → stratified sampling → kiểm tra tính đại diện → lưu sop_20k.csv`

Sau notebook này, output chính là:

```text
data/splits/sop_20k.csv
```

Ngoài ra tạo:

```text
outputs/sampling/
├── class_distribution.csv
├── sampling_summary.json
├── sampling_distribution.png
└── sample_preview.png
```

### Nguyên tắc

- Stratify theo `class_id` vì đây là **product identity** dùng làm ground-truth retrieval.
- Không lấy random 20K đơn thuần.
- Ưu tiên giữ tỷ lệ xuất hiện của các class.
- Chỉ giữ class có ít nhất 2 ảnh để có thể tạo query/gallery ở bước evaluation.
- Không thay đổi `class_id`, `super_class_id`, `image_id` và `path`.

## Cell 1 — Cấu hình experiment

In [ ]:
from pathlib import Path

# =========================
# USER CONFIGURATION
# =========================

# Nếu chạy notebook trong repo:
PROJECT_ROOT = Path("..").resolve()

# Nếu notebook nằm trong visual_product_search/notebooks/
SOP_ROOT = PROJECT_ROOT / "data" / "raw" / "Stanford_Online_Products"

TRAIN_FILE = SOP_ROOT / "Ebay_train.txt"
TEST_FILE = SOP_ROOT / "Ebay_test.txt"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "sampling"
SPLIT_DIR = PROJECT_ROOT / "data" / "splits"

SAMPLE_N = 20_000
RANDOM_SEED = 42

# Có kiểm tra image tồn tại hay không.
CHECK_IMAGE_EXISTS = True

# Nếu True: chỉ lấy class có >= 2 ảnh.
# Khuyến nghị giữ True cho retrieval experiment.
REQUIRE_MIN_2_IMAGES_PER_CLASS = True

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SOP_ROOT:", SOP_ROOT)
print("SAMPLE_N:", SAMPLE_N)
print("RANDOM_SEED:", RANDOM_SEED)

## Cell 2 — Import thư viện

In [ ]:
import json
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter

print("NumPy :", np.__version__)
print("Pandas:", pd.__version__)

## Cell 3 — Kiểm tra file SOP

In [ ]:
print("Train metadata:", TRAIN_FILE)
print("Test metadata :", TEST_FILE)

print("\nTrain exists:", TRAIN_FILE.exists())
print("Test exists :", TEST_FILE.exists())

if not TRAIN_FILE.exists() or not TEST_FILE.exists():
    raise FileNotFoundError(
        "Không tìm thấy Ebay_train.txt/Ebay_test.txt. "
        "Hãy kiểm tra SOP_ROOT ở Cell 1."
    )

## Cell 4 — Đọc `Ebay_train.txt` và `Ebay_test.txt`

Theo format SOP:

```text
image_id class_id super_class_id path
```

Ví dụ:

```text
1 1 1 bicycle_final/111085122871_0.JPG
```

Ta giữ nguyên metadata gốc và thêm `source_split`.

In [ ]:
def read_sop_metadata(txt_path, source_split):
    rows = []

    with open(txt_path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()

            if not line:
                continue

            parts = line.split(maxsplit=3)

            if len(parts) != 4:
                print(f"Warning: malformed line {line_no}: {line}")
                continue

            image_id, class_id, super_class_id, path = parts

            rows.append({
                "image_id": int(image_id),
                "class_id": int(class_id),
                "super_class_id": int(super_class_id),
                "path": path,
                "source_split": source_split,
            })

    return pd.DataFrame(rows)


train_df = read_sop_metadata(TRAIN_FILE, "train")
test_df = read_sop_metadata(TEST_FILE, "test")

print("Train:", train_df.shape)
print("Test :", test_df.shape)

display(train_df.head())

## Cell 5 — Gộp metadata

In [ ]:
full_df = pd.concat(
    [train_df, test_df],
    ignore_index=True
)

print("Total images:", len(full_df))
print("Unique image IDs:", full_df.image_id.nunique())
print("Unique product classes:", full_df.class_id.nunique())
print("Unique super classes:", full_df.super_class_id.nunique())

display(full_df.head())

## Cell 6 — Kiểm tra duplicate metadata

In [ ]:
print("Duplicate image_id:",
      full_df.image_id.duplicated().sum())

print("Duplicate path:",
      full_df.path.duplicated().sum())

# Kiểm tra một image_id có bị gán nhiều class không.
class_per_image = full_df.groupby("image_id")["class_id"].nunique()

print(
    "Image IDs with multiple class_id:",
    (class_per_image > 1).sum()
)

## Cell 7 — Tạo absolute image path

In [ ]:
full_df["image_path"] = full_df["path"].map(
    lambda p: str(SOP_ROOT / p)
)

display(
    full_df[
        ["image_id", "class_id", "super_class_id",
         "path", "image_path", "source_split"]
    ].head()
)

## Cell 8 — Kiểm tra image có thực sự tồn tại

In [ ]:
if CHECK_IMAGE_EXISTS:
    full_df["image_exists"] = full_df["image_path"].map(
        os.path.exists
    )

    missing = (~full_df["image_exists"]).sum()

    print("Missing images:", missing)
    print("Existing ratio:",
          full_df["image_exists"].mean())

    if missing > 0:
        display(
            full_df.loc[
                ~full_df["image_exists"],
                ["image_id", "class_id", "path", "image_path"]
            ].head(20)
        )
else:
    full_df["image_exists"] = True
    print("Image existence check skipped.")

## Cell 9 — Chọn tập candidate trước sampling

In [ ]:
candidate_df = full_df.copy()

if CHECK_IMAGE_EXISTS:
    candidate_df = candidate_df[
        candidate_df["image_exists"]
    ].copy()

if REQUIRE_MIN_2_IMAGES_PER_CLASS:
    class_counts = candidate_df.groupby("class_id").size()

    valid_classes = class_counts[
        class_counts >= 2
    ].index

    candidate_df = candidate_df[
        candidate_df["class_id"].isin(valid_classes)
    ].copy()

print("Candidate images:", len(candidate_df))
print("Candidate classes:", candidate_df.class_id.nunique())

class_counts = candidate_df.groupby("class_id").size()

print("\nClass size statistics:")
display(class_counts.describe())

## Cell 10 — Phân tích phân bố class trước sampling

In [ ]:
class_stats_before = (
    candidate_df
    .groupby(["class_id", "super_class_id"])
    .size()
    .reset_index(name="n_images")
    .sort_values("n_images", ascending=False)
)

display(class_stats_before.head(20))

plt.figure(figsize=(8, 5))
plt.hist(
    class_stats_before["n_images"],
    bins=50
)
plt.xlabel("Number of images per class")
plt.ylabel("Number of classes")
plt.title("SOP class-size distribution before sampling")
plt.grid(alpha=0.2)
plt.show()

## Cell 11 — Hàm tính allocation cho stratified sampling

In [ ]:
def proportional_allocation(
    df,
    target_n,
    group_col="class_id",
    min_per_group=2,
):
    """
    Allocate target_n samples approximately proportional
    to each class size.

    Constraints:
    - each selected class gets at least min_per_group
    - no class receives more samples than it owns
    """

    counts = df[group_col].value_counts().sort_index()

    if min_per_group > 0:
        counts = counts[counts >= min_per_group]

    if len(counts) == 0:
        raise ValueError("Không có class hợp lệ.")

    if target_n < len(counts) * min_per_group:
        raise ValueError(
            f"SAMPLE_N={target_n} quá nhỏ. "
            f"Cần ít nhất {len(counts) * min_per_group}."
        )

    # Initial proportional allocation
    raw = counts / counts.sum() * target_n

    allocation = np.floor(raw).astype(int)

    # Minimum samples per selected class
    allocation = allocation.clip(lower=min_per_group)

    # Không vượt quá số ảnh thật
    allocation = pd.Series(
        np.minimum(allocation, counts),
        index=counts.index,
        dtype=int
    )

    # Điều chỉnh tổng về đúng target_n
    while allocation.sum() < target_n:
        capacity = counts - allocation
        available = capacity[capacity > 0]

        if len(available) == 0:
            break

        # Ưu tiên class còn thiếu nhiều theo proportional raw target
        deficit = raw - allocation
        candidates = deficit.loc[available.index]

        selected_class = candidates.idxmax()
        allocation.loc[selected_class] += 1

    while allocation.sum() > target_n:
        removable = allocation[
            allocation > min_per_group
        ]

        if len(removable) == 0:
            break

        excess = allocation - raw
        selected_class = excess.loc[
            removable.index
        ].idxmax()

        allocation.loc[selected_class] -= 1

    return counts, allocation


original_counts, allocation = proportional_allocation(
    candidate_df,
    target_n=SAMPLE_N,
    group_col="class_id",
    min_per_group=2 if REQUIRE_MIN_2_IMAGES_PER_CLASS else 1,
)

print("Target:", SAMPLE_N)
print("Allocated:", allocation.sum())
print("Selected classes:", len(allocation))

display(
    pd.DataFrame({
        "original_count": original_counts,
        "allocated_count": allocation,
    }).head(20)
)

## Cell 12 — Kiểm tra allocation

In [ ]:
allocation_check = pd.DataFrame({
    "original_count": original_counts,
    "sample_count": allocation,
})

allocation_check["sample_ratio"] = (
    allocation_check["sample_count"]
    / allocation_check["sample_count"].sum()
)

allocation_check["original_ratio"] = (
    allocation_check["original_count"]
    / allocation_check["original_count"].sum()
)

allocation_check["ratio_diff"] = (
    allocation_check["sample_ratio"]
    - allocation_check["original_ratio"]
)

print("Total allocated:", allocation_check.sample_count.sum())
print(
    "Classes with zero allocation:",
    (allocation_check.sample_count == 0).sum()
)

display(allocation_check.describe())

## Cell 13 — Thực hiện stratified sampling

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)

sample_parts = []

for class_id, n_samples in allocation.items():

    class_df = candidate_df[
        candidate_df["class_id"] == class_id
    ]

    selected_indices = rng.choice(
        len(class_df),
        size=int(n_samples),
        replace=False
    )

    selected = class_df.iloc[selected_indices].copy()

    sample_parts.append(selected)

sample_df = pd.concat(
    sample_parts,
    ignore_index=True
)

# Shuffle toàn bộ sample
sample_df = sample_df.sample(
    frac=1,
    random_state=RANDOM_SEED
).reset_index(drop=True)

print("Sample size:", len(sample_df))
print("Sample classes:", sample_df.class_id.nunique())

## Cell 14 — Kiểm tra sample size và uniqueness

In [ ]:
assert len(sample_df) == SAMPLE_N, (
    f"Expected {SAMPLE_N}, got {len(sample_df)}"
)

assert sample_df.image_id.nunique() == len(sample_df), (
    "Duplicate image_id trong sample."
)

print("✓ Sample size đúng:", len(sample_df))
print("✓ Image IDs unique")
print("✓ Classes:", sample_df.class_id.nunique())

## Cell 15 — So sánh phân bố original vs sampled

In [ ]:
sample_counts = sample_df.groupby("class_id").size()

distribution = pd.DataFrame({
    "original_count": original_counts,
    "sample_count": sample_counts,
}).fillna(0)

distribution["original_ratio"] = (
    distribution["original_count"]
    / distribution["original_count"].sum()
)

distribution["sample_ratio"] = (
    distribution["sample_count"]
    / distribution["sample_count"].sum()
)

distribution["ratio_diff"] = (
    distribution["sample_ratio"]
    - distribution["original_ratio"]
)

display(distribution.head(20))

## Cell 16 — Định lượng mức độ giữ phân bố

In [ ]:
# Mean absolute difference giữa tỷ lệ class
mad = distribution["ratio_diff"].abs().mean()

# Maximum absolute difference
max_diff = distribution["ratio_diff"].abs().max()

# Correlation giữa số ảnh original và số ảnh sampled
corr = distribution[
    ["original_count", "sample_count"]
].corr().iloc[0, 1]

print(f"Mean absolute ratio difference : {mad:.8f}")
print(f"Maximum absolute ratio diff   : {max_diff:.8f}")
print(f"Count correlation              : {corr:.6f}")

## Cell 17 — Vẽ original vs sampled distribution

In [ ]:
plt.figure(figsize=(7, 6))

plt.scatter(
    distribution["original_ratio"],
    distribution["sample_ratio"],
    s=12
)

mx = max(
    distribution["original_ratio"].max(),
    distribution["sample_ratio"].max()
)

plt.plot(
    [0, mx],
    [0, mx],
    linestyle="--"
)

plt.xlabel("Original class proportion")
plt.ylabel("Sampled class proportion")
plt.title("Stratified sampling: original vs sampled")
plt.grid(alpha=0.2)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

fig_path = OUTPUT_DIR / "sampling_distribution.png"
plt.savefig(fig_path, dpi=200, bbox_inches="tight")

plt.show()

print("Saved:", fig_path)

## Cell 18 — Kiểm tra phân bố theo `super_class_id`

In [ ]:
super_before = (
    candidate_df.groupby("super_class_id")
    .size()
    .rename("original_count")
)

super_after = (
    sample_df.groupby("super_class_id")
    .size()
    .rename("sample_count")
)

super_dist = pd.concat(
    [super_before, super_after],
    axis=1
).fillna(0)

super_dist["original_ratio"] = (
    super_dist["original_count"]
    / super_dist["original_count"].sum()
)

super_dist["sample_ratio"] = (
    super_dist["sample_count"]
    / super_dist["sample_count"].sum()
)

super_dist["ratio_diff"] = (
    super_dist["sample_ratio"]
    - super_dist["original_ratio"]
)

display(super_dist.sort_values(
    "original_count",
    ascending=False
).head(20))

print(
    "Super-class mean absolute ratio difference:",
    super_dist["ratio_diff"].abs().mean()
)

## Cell 19 — Kiểm tra train/test source split trong sample

In [ ]:
split_distribution = (
    sample_df["source_split"]
    .value_counts()
    .rename_axis("source_split")
    .reset_index(name="count")
)

split_distribution["ratio"] = (
    split_distribution["count"]
    / split_distribution["count"].sum()
)

display(split_distribution)

## Cell 20 — Preview ảnh được sampling

In [ ]:
from PIL import Image

preview = sample_df.sample(
    min(9, len(sample_df)),
    random_state=RANDOM_SEED
)

plt.figure(figsize=(12, 12))

for i, (_, row) in enumerate(preview.iterrows(), start=1):
    ax = plt.subplot(3, 3, i)

    try:
        image = Image.open(
            row["image_path"]
        ).convert("RGB")

        ax.imshow(image)

        ax.set_title(
            f"image={row.image_id}\n"
            f"class={row.class_id}\n"
            f"super={row.super_class_id}"
        )

    except Exception as e:
        ax.set_title(f"ERROR\n{e}")

    ax.axis("off")

plt.tight_layout()

preview_path = OUTPUT_DIR / "sample_preview.png"
plt.savefig(
    preview_path,
    dpi=200,
    bbox_inches="tight"
)

plt.show()

print("Saved:", preview_path)

## Cell 21 — Lưu `sop_20k.csv`

In [ ]:
SPLIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

sample_output = SPLIT_DIR / "sop_20k.csv"

# Chỉ lưu những field cần thiết cho các notebook tiếp theo.
columns = [
    "image_id",
    "class_id",
    "super_class_id",
    "path",
    "image_path",
    "source_split",
]

sample_df[columns].to_csv(
    sample_output,
    index=False
)

print("Saved:", sample_output)
print("Rows:", len(sample_df))

## Cell 22 — Lưu bảng thống kê sampling

In [ ]:
distribution_output = (
    OUTPUT_DIR / "class_distribution.csv"
)

distribution.reset_index(
    names="class_id"
).to_csv(
    distribution_output,
    index=False
)

print("Saved:", distribution_output)

## Cell 23 — Lưu summary JSON

In [ ]:
summary = {
    "source": "Stanford Online Products",
    "sampling_method": "proportional stratified sampling",
    "stratification_key": "class_id",
    "original_images": int(len(full_df)),
    "candidate_images": int(len(candidate_df)),
    "sample_images": int(len(sample_df)),
    "original_classes": int(full_df.class_id.nunique()),
    "candidate_classes": int(candidate_df.class_id.nunique()),
    "sample_classes": int(sample_df.class_id.nunique()),
    "random_seed": RANDOM_SEED,
    "min_images_per_selected_class": (
        int(sample_df.groupby("class_id").size().min())
    ),
    "mean_absolute_class_ratio_difference": float(mad),
    "maximum_class_ratio_difference": float(max_diff),
    "class_count_correlation": float(corr),
}

summary_path = OUTPUT_DIR / "sampling_summary.json"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False
    )

print(json.dumps(
    summary,
    indent=2,
    ensure_ascii=False
))

print("\nSaved:", summary_path)

## Cell 24 — Final validation

Notebook 01 được xem là **hoàn thành** nếu toàn bộ assertion dưới đây pass:

- `sop_20k.csv` có đúng 20,000 ảnh.
- Không duplicate `image_id`.
- Mỗi selected `class_id` có ít nhất 2 ảnh.
- Tất cả image path đều tồn tại nếu bật `CHECK_IMAGE_EXISTS`.
- Phân bố class sau sampling gần với phân bố ban đầu.
- Metadata gốc (`image_id`, `class_id`, `super_class_id`, `path`) được giữ nguyên.

Output này sẽ được notebook 02/03 sử dụng, **không sampling lại**.

In [ ]:
# Final validation
saved = pd.read_csv(sample_output)

assert len(saved) == SAMPLE_N
assert saved.image_id.nunique() == SAMPLE_N

if CHECK_IMAGE_EXISTS:
    assert saved.image_path.map(os.path.exists).all()

class_sample_counts = saved.groupby("class_id").size()

if REQUIRE_MIN_2_IMAGES_PER_CLASS:
    assert class_sample_counts.min() >= 2

required_columns = {
    "image_id",
    "class_id",
    "super_class_id",
    "path",
    "image_path",
    "source_split",
}

assert required_columns.issubset(saved.columns)

print("=" * 60)
print("SAMPLING NOTEBOOK COMPLETED SUCCESSFULLY")
print("=" * 60)
print("Output:", sample_output)
print("Images:", len(saved))
print("Classes:", saved.class_id.nunique())
print("Min images/class:", class_sample_counts.min())
print("Max images/class:", class_sample_counts.max())
print("All image paths exist:", saved.image_path.map(os.path.exists).all())

# Expected output

Sau khi chạy xong notebook, thư mục nên có:

```text
data/
└── splits/
    └── sop_20k.csv

outputs/
└── sampling/
    ├── class_distribution.csv
    ├── sampling_summary.json
    ├── sampling_distribution.png
    └── sample_preview.png
```

`notebooks/02_baseline.ipynb` sẽ **đọc `data/splits/sop_20k.csv`**, không thực hiện sampling lại.